<a href="https://colab.research.google.com/github/mennasherif14/Chicago-Traffic-ETL/blob/main/Chicago_Traffic_Crashes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: ETL Pipeline with Open Data (Chicago Traffic Crashes)
## Milestone 1: Data Collection, Exploration & Preprocessing

**Objective**:  
Ingest raw CSV from Chicago Open Data → Explore data quality → Clean & transform using **Polars** → Save as optimized **Parquet** file.

**Dataset**: Traffic Crashes - Crashes (City of Chicago)  
- ~1.07 Million rows, 49 columns  
- Real-world messy data (missing values, date formats, categoricals)

**Tools**: Polars (fast, memory-efficient), Pandas (for quick previews)


**All required steps covered**:
- Import libraries
- Load data
- Initial inspection
- Remove duplicates
- Handle missing values
- Handle data types
- Feature Engineering
- Filter and clean data
- Handle outliers
- Aggregate data
- Validate data
- Reduce data size
- Export clean data
- Document the cleaning process

**Dataset**: Chicago Traffic Crashes (Crashes table)

In [1]:
!pip install polars[lazy,pandas] pyarrow -q

import polars as pl
import pandas as pd
import numpy as np
import os
import requests
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)

print("Libraries imported successfully")
print("Polars version:", pl.__version__)

Libraries imported successfully
Polars version: 1.35.2


# **Data Collection - Download CSV**

In [3]:
csv_url = "https://data.cityofchicago.org/api/views/85ca-t3if/rows.csv?accessType=DOWNLOAD"
data_dir = "/content/data"
os.makedirs(data_dir, exist_ok=True)
csv_path = f"{data_dir}/chicago_traffic_crashes_raw.csv"

if not os.path.exists(csv_path):
    print("Downloading dataset (~500+ MB)...")
    r = requests.get(csv_url, stream=True)
    with open(csv_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=16*1024*1024):
            f.write(chunk)
    print("Download complete")
else:
    print("Using existing file")

# Lazy load for efficiency
lf = pl.scan_csv(csv_path, infer_schema_length=100_000, low_memory=True, null_values=["", "NULL", "NA"])
print(f"Total rows: {lf.select(pl.len()).collect().item():,}")

Download complete
Total rows: 1,067,469


# **Initial Data Loading & Schema Exploration**

In [8]:
# Debug: Confirm current state
print("Type of lf:", type(lf))
print("Is LazyFrame?", isinstance(lf, pl.LazyFrame))

# Re-create lf safely if needed
lf = pl.scan_csv(
    csv_path,
    infer_schema_length=100_000,
    low_memory=True,
    null_values=["", "NULL", "NA", "UNKNOWN"]
)
print("lf re-initialized as LazyFrame")

Type of lf: <class 'polars.lazyframe.frame.LazyFrame'>
Is LazyFrame? True
lf re-initialized as LazyFrame


In [10]:
# === Initial Inspection (Fully Robust) ===

print("=== Schema ===")
print(lf.collect_schema())

print("\n=== First 5 rows ===")
display(lf.head(5).collect())

print("\n=== Summary Statistics ===")
# Force LazyFrame behavior
summary_lf = lf.describe()
summary = summary_lf.collect() if isinstance(summary_lf, pl.LazyFrame) else summary_lf
display(summary)

print("\n=== Column Overview (Sample-based for speed) ===")
# Use a sample to compute unique counts quickly
sample_df = lf.head(50_000).collect()   # 50k is enough for good stats

for col in lf.columns:
    dtype = lf.collect_schema()[col]
    unique = sample_df[col].n_unique()
    null_pct = round((sample_df[col].is_null().sum() / len(sample_df) * 100), 2)
    print(f"{col:40} | {str(dtype):12} | Unique: {unique:,} | Null%: {null_pct}%")

=== Schema ===
Schema({'CRASH_RECORD_ID': String, 'CRASH_DATE_EST_I': String, 'CRASH_DATE': String, 'POSTED_SPEED_LIMIT': Int64, 'TRAFFIC_CONTROL_DEVICE': String, 'DEVICE_CONDITION': String, 'WEATHER_CONDITION': String, 'LIGHTING_CONDITION': String, 'FIRST_CRASH_TYPE': String, 'TRAFFICWAY_TYPE': String, 'LANE_CNT': Int64, 'ALIGNMENT': String, 'ROADWAY_SURFACE_COND': String, 'ROAD_DEFECT': String, 'REPORT_TYPE': String, 'CRASH_TYPE': String, 'INTERSECTION_RELATED_I': String, 'NOT_RIGHT_OF_WAY_I': String, 'HIT_AND_RUN_I': String, 'DAMAGE': String, 'DATE_POLICE_NOTIFIED': String, 'PRIM_CONTRIBUTORY_CAUSE': String, 'SEC_CONTRIBUTORY_CAUSE': String, 'STREET_NO': Int64, 'STREET_DIRECTION': String, 'STREET_NAME': String, 'BEAT_OF_OCCURRENCE': Int64, 'PHOTOS_TAKEN_I': String, 'STATEMENTS_TAKEN_I': String, 'DOORING_I': String, 'WORK_ZONE_I': String, 'WORK_ZONE_TYPE': String, 'WORKERS_PRESENT_I': String, 'NUM_UNITS': Int64, 'CRASH_MONTH': Int64, 'MOST_SEVERE_INJURY': String, 'INJURIES_TOTAL': In

CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,LANE_CNT,ALIGNMENT,ROADWAY_SURFACE_COND,ROAD_DEFECT,REPORT_TYPE,CRASH_TYPE,INTERSECTION_RELATED_I,NOT_RIGHT_OF_WAY_I,HIT_AND_RUN_I,DAMAGE,DATE_POLICE_NOTIFIED,PRIM_CONTRIBUTORY_CAUSE,SEC_CONTRIBUTORY_CAUSE,STREET_NO,STREET_DIRECTION,STREET_NAME,BEAT_OF_OCCURRENCE,PHOTOS_TAKEN_I,STATEMENTS_TAKEN_I,DOORING_I,WORK_ZONE_I,WORK_ZONE_TYPE,WORKERS_PRESENT_I,NUM_UNITS,CRASH_MONTH,MOST_SEVERE_INJURY,INJURIES_TOTAL,INJURIES_FATAL,INJURIES_INCAPACITATING,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,IDOT_CONTROL_NO,LATITUDE,LONGITUDE,LOCATION
str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64,f64,str
"""006bc88aa00209dba00e7f4e77939f4cac6d1918b26e0fdc14…",null,"""07/28/2020 09:37:00 PM""",25,"""NO CONTROLS""","""NO CONTROLS""","""CLEAR""","""DARKNESS""","""FIXED OBJECT""","""NOT DIVIDED""",null,"""CURVE, LEVEL""","""DRY""","""NO DEFECTS""","""ON SCENE""","""INJURY AND / OR TOW DUE TO CRASH""",null,null,null,"""OVER $1,500""","""07/28/2020 10:38:00 PM""","""PHYSICAL CONDITION OF DRIVER""","""UNABLE TO DETERMINE""",8300,"""S""","""COMMERCIAL AVE""",424,null,null,null,null,null,null,1,7,"""NONINCAPACITATING INJURY""",1,0,0,1,0,3,0,21,3,"""X001971047""",null,null,null
"""8bd0613f9b39d21c255f4c0c6964a95418c448fe25015be83f…",null,"""08/12/2020 06:49:00 PM""",30,"""TRAFFIC SIGNAL""","""FUNCTIONING PROPERLY""","""CLEAR""","""DUSK""","""SIDESWIPE SAME DIRECTION""","""FOUR WAY""",null,"""STRAIGHT AND LEVEL""","""DRY""","""NO DEFECTS""","""NOT ON SCENE (DESK REPORT)""","""NO INJURY / DRIVE AWAY""","""Y""",null,"""Y""","""$501 - $1,500""","""08/12/2020 09:00:00 PM""","""IMPROPER OVERTAKING/PASSING""","""IMPROPER OVERTAKING/PASSING""",2400,"""E""","""71ST ST""",334,null,null,null,null,null,null,2,8,"""NO INDICATION OF INJURY""",0,0,0,0,0,2,0,18,4,"""X001984876""",null,null,null
"""54bb43f31434a799998791acfc9020eb0a09401332a08ed9a7…",null,"""08/09/2020 01:55:00 AM""",25,"""NO CONTROLS""","""NO CONTROLS""","""CLEAR""","""DARKNESS, LIGHTED ROAD""","""HEAD ON""","""NOT DIVIDED""",null,"""STRAIGHT AND LEVEL""","""DRY""","""NO DEFECTS""","""ON SCENE""","""INJURY AND / OR TOW DUE TO CRASH""",null,null,null,"""OVER $1,500""","""08/09/2020 01:55:00 AM""","""UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARR…","""NOT APPLICABLE""",551,"""S""","""MILLARD AVE""",1133,null,null,null,null,null,null,2,8,"""NONINCAPACITATING INJURY""",5,0,0,5,0,0,0,1,1,"""X001981337""",null,null,null
"""3d8f4ecfbdce63facb0497b9b00b4538e0f75fe7de3ad81d34…",null,"""07/27/2020 09:30:00 AM""",30,"""NO CONTROLS""","""NO CONTROLS""","""RAIN""","""DAYLIGHT""","""PARKED MOTOR VEHICLE""","""ONE-WAY""",null,"""STRAIGHT AND LEVEL""","""WET""","""NO DEFECTS""","""ON SCENE""","""NO INJURY / DRIVE AWAY""",null,null,null,"""$501 - $1,500""","""07/27/2020 09:30:00 AM""","""WEATHER""","""NOT APPLICABLE""",5329,"""S""","""INDIANA AVE""",231,null,null,null,null,null,null,2,7,"""NO INDICATION OF INJURY""",0,0,0,0,0,1,0,9,2,null,41.79746,-87.620768,"""POINT (-87.62076816217 41.797459871568)"""
"""b454dcc68b7f1cc9dbae0e8f8caa438d184ded31584554f1b9…",null,"""08/10/2020 04:30:00 PM""",30,"""NO CONTROLS""","""OTHER""","""RAIN""","""DAYLIGHT""","""OTHER NONCOLLISION""","""OTHER""",null,"""STRAIGHT AND LEVEL""","""WET""","""OTHER""","""NOT ON SCENE (DESK REPORT)""","""NO INJURY / DRIVE AWAY""",null,null,null,"""OVER $1,500""","""08/11/2020 01:40:00 PM""","""WEATHER""","""NOT APPLICABLE""",5547,"""S""","""NEVA AVE""",811,null,null,null,null,null,null,1,8,null,null,null,null,null,null,null,null,16,2,"""X001983368""",41.790535,-87.800059,"""POINT (-87.800059225607 41.790535405928)"""



=== Summary Statistics ===


statistic,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,LANE_CNT,ALIGNMENT,ROADWAY_SURFACE_COND,ROAD_DEFECT,REPORT_TYPE,CRASH_TYPE,INTERSECTION_RELATED_I,NOT_RIGHT_OF_WAY_I,HIT_AND_RUN_I,DAMAGE,DATE_POLICE_NOTIFIED,PRIM_CONTRIBUTORY_CAUSE,SEC_CONTRIBUTORY_CAUSE,STREET_NO,STREET_DIRECTION,STREET_NAME,BEAT_OF_OCCURRENCE,PHOTOS_TAKEN_I,STATEMENTS_TAKEN_I,DOORING_I,WORK_ZONE_I,WORK_ZONE_TYPE,WORKERS_PRESENT_I,NUM_UNITS,CRASH_MONTH,MOST_SEVERE_INJURY,INJURIES_TOTAL,INJURIES_FATAL,INJURIES_INCAPACITATING,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,IDOT_CONTROL_NO,LATITUDE,LONGITUDE,LOCATION
str,str,str,str,f64,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,f64,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str
"""count""","""1067469""","""77177""","""1067469""",1.067469e6,"""1017226""","""986355""","""1001761""","""1014090""","""1067469""","""1054538""",199044.0,"""1067469""","""963326""","""852088""","""1031393""","""1067469""","""245255""","""47598""","""335015""","""1067469""","""1067469""","""1067469""","""1067469""",1.067469e6,"""1067465""","""1067467""",1.067464e6,"""15443""","""25864""","""3392""","""5727""","""3734""","""1469""",1.067469e6,1.067469e6,"""1065149""",1.065164e6,1.065164e6,1.065164e6,1.065164e6,1.065164e6,1.065164e6,1.065164e6,1.067469e6,1.067469e6,"""1064663""",1.059213e6,1.059213e6,"""1059213"""
"""null_count""","""0""","""990292""","""0""",0.0,"""50243""","""81114""","""65708""","""53379""","""0""","""12931""",868425.0,"""0""","""104143""","""215381""","""36076""","""0""","""822214""","""1019871""","""732454""","""0""","""0""","""0""","""0""",0.0,"""4""","""2""",5.0,"""1052026""","""1041605""","""1064077""","""1061742""","""1063735""","""1066000""",0.0,0.0,"""2320""",2305.0,2305.0,2305.0,2305.0,2305.0,2305.0,2305.0,0.0,0.0,"""2806""",8256.0,8256.0,"""8256"""
"""mean""",null,null,null,28.42129,null,null,null,null,null,null,13.328204,null,null,null,null,null,null,null,null,null,null,null,null,3685.425288,null,null,1248.612436,null,null,null,null,null,null,2.034986,6.586833,null,0.19952,0.001135,0.018992,0.110186,0.069207,1.995315,0.0,13.188598,4.11854,null,41.855202,-87.673072,null
"""std""",null,null,null,6.021975,null,null,null,null,null,null,2961.340758,null,null,null,null,null,null,null,null,null,null,null,null,2869.930626,null,null,704.505571,null,null,null,null,null,null,0.448647,3.404565,null,0.580156,0.036584,0.161162,0.427474,0.339193,1.152952,0.0,5.572378,1.978548,null,0.358006,0.730256,null
"""min""","""000013b0123279411e0ec856dae95ab9f0851764350b7feaeb…","""N""","""01/01/2016 01:00:00 AM""",0.0,"""BICYCLE CROSSING SIGN""","""FUNCTIONING IMPROPERLY""","""BLOWING SAND, SOIL, DIRT""","""DARKNESS""","""ANGLE""","""ALLEY""",0.0,"""CURVE ON GRADE""","""DRY""","""DEBRIS ON ROADWAY""","""AMENDED""","""INJURY AND / OR TOW DUE TO CRASH""","""N""","""N""","""N""","""$500 OR LESS""","""01/01/2016 01:00:00 PM""","""ANIMAL""","""ANIMAL""",0.0,"""E""","""100TH DR""",111.0,"""N""","""N""","""N""","""N""","""CONSTRUCTION""","""N""",1.0,1.0,"""FATAL""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""X000288231""",0.0,-87.939678,"""POINT (-87.524587334823 41.702790638503)"""
"""25%""",null,null,null,30.0,null,null,null,null,null,null,2.0,null,null,null,null,null,null,null,null,null,null,null,null,1258.0,null,null,722.0,null,null,null,null,null,null,2.0,4.0,null,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,2.0,null,41.784333,-87.721824,null
"""50%""",null,null,null,30.0,null,null,null,null,null,null,2.0,null,null,null,null,null,null,null,null,null,null,null,null,3201.0,null,null,1213.0,null,null,null,null,null,null,2.0,7.0,null,0.0,0.0,0.0,0.0,0.0,2.0,0.0,14.0,4.0,null,41.87519,-87.674458,null
"""75%""",null,null,null,30.0,null,null,null,null,null,null,4.0,n


=== Column Overview (Sample-based for speed) ===
CRASH_RECORD_ID                          | String       | Unique: 50,000 | Null%: 0.0%
CRASH_DATE_EST_I                         | String       | Unique: 3 | Null%: 92.75%
CRASH_DATE                               | String       | Unique: 48,519 | Null%: 0.0%
POSTED_SPEED_LIMIT                       | Int64        | Unique: 25 | Null%: 0.0%
TRAFFIC_CONTROL_DEVICE                   | String       | Unique: 19 | Null%: 4.76%
DEVICE_CONDITION                         | String       | Unique: 8 | Null%: 7.71%
WEATHER_CONDITION                        | String       | Unique: 12 | Null%: 6.07%
LIGHTING_CONDITION                       | String       | Unique: 6 | Null%: 4.83%
FIRST_CRASH_TYPE                         | String       | Unique: 18 | Null%: 0.0%
TRAFFICWAY_TYPE                          | String       | Unique: 20 | Null%: 1.3%
LANE_CNT                                 | Int64        | Unique: 17 | Null%: 81.83%
ALIGNMENT               

# **Remove Duplicates**

In [25]:
print("=== DUPLICATE REMOVAL ===")

# Count before
rows_before = lf.select(pl.len()).collect().item()
print(f"Rows BEFORE removing duplicates: {rows_before:,}")

# Remove duplicates using the unique primary key
clean_lf = lf.unique(
    subset=["CRASH_RECORD_ID"],
    keep="first"
)

rows_after = clean_lf.select(pl.len()).collect().item()
print(f"Rows AFTER removing duplicates : {rows_after:,}")
print(f"Duplicates removed            : {rows_before - rows_after:,}")

=== DUPLICATE REMOVAL ===
Rows BEFORE removing duplicates: 1,067,469
Rows AFTER removing duplicates : 1,067,469
Duplicates removed            : 0


# **Handling Missing Values**

In [14]:
print("=== Missing Values Analysis ===")
# Sample for speed
sample_missing = clean_lf.head(100_000).collect()

missing_stats = (
    sample_missing
    .select([pl.col(c).is_null().sum().alias(c) for c in sample_missing.columns])
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
    .with_columns(
        (pl.col("null_count") / len(sample_missing) * 100).round(2).alias("null_pct")
    )
)

display(missing_stats.sort("null_count", descending=True).head(15))

# === Apply Missing Value Handling ===
clean_lf = (
    clean_lf
    # Drop rows with missing critical location
    .filter(pl.col("LATITUDE").is_not_null() & pl.col("LONGITUDE").is_not_null())

    # Convert common unknown values to null
    .with_columns([
        pl.when(pl.col(c).is_in(["UNKNOWN", "OTHER", "", "NOT APPLICABLE"]))
          .then(None)
          .otherwise(pl.col(c))
          .alias(c)
        for c in ["WEATHER_CONDITION", "LIGHTING_CONDITION", "ROADWAY_SURFACE_COND",
                  "PRIM_CONTRIBUTORY_CAUSE", "SEC_CONTRIBUTORY_CAUSE", "FIRST_CRASH_TYPE",
                  "MOST_SEVERE_INJURY"]
    ])

    # Fill numeric columns
    .with_columns([
        pl.col("POSTED_SPEED_LIMIT").fill_null(30),
        pl.col(["INJURIES_TOTAL", "INJURIES_FATAL", "INJURIES_INCAPACITATING",
                "INJURIES_NON_INCAPACITATING", "NUM_UNITS"]).fill_null(0)
    ])
)

print("Rows after missing value handling:", clean_lf.select(pl.len()).collect().item())

=== Missing Values Analysis ===


column,null_count,null_pct
str,u32,f64
"""WORKERS_PRESENT_I""",99866,99.87
"""DOORING_I""",99721,99.72
"""WORK_ZONE_TYPE""",99654,99.65
"""WORK_ZONE_I""",99472,99.47
"""PHOTOS_TAKEN_I""",98506,98.51
…,…,…
"""HIT_AND_RUN_I""",68950,68.95
"""ROAD_DEFECT""",20032,20.03
"""ROADWAY_SURFACE_COND""",9659,9.66


Rows after missing value handling: 1059213


# **Handle Data Types + Feature Engineering**

In [15]:
print("=== Data Type Handling + Feature Engineering ===")

clean_lf = (
    clean_lf
    # Date conversion
    .with_columns([
        pl.col("CRASH_DATE").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p", strict=False)
          .alias("CRASH_DATETIME")
    ])
    .with_columns([
        pl.col("CRASH_DATETIME").dt.date().alias("CRASH_DATE_ONLY"),
        pl.col("CRASH_DATETIME").dt.year().alias("CRASH_YEAR"),
        pl.col("CRASH_DATETIME").dt.month().alias("CRASH_MONTH_NEW"),
        pl.col("CRASH_DATETIME").dt.hour().alias("CRASH_HOUR_NEW"),
        pl.col("CRASH_DATETIME").dt.weekday().alias("CRASH_WEEKDAY")  # 0 = Monday
    ])

    # Cast types for efficiency
    .with_columns([
        pl.col("POSTED_SPEED_LIMIT").cast(pl.Int16),
        pl.col(["INJURIES_TOTAL", "INJURIES_FATAL", "INJURIES_INCAPACITATING",
                "INJURIES_NON_INCAPACITATING", "NUM_UNITS", "BEAT_OF_OCCURRENCE"]).cast(pl.Int16),
        pl.col(["LATITUDE", "LONGITUDE"]).cast(pl.Float32),
    ])

    # New engineered features
    .with_columns([
        (pl.col("INJURIES_TOTAL") > 0).alias("HAD_INJURIES"),
        (pl.col("INJURIES_FATAL") > 0).alias("HAD_FATALITY"),
        pl.when(pl.col("CRASH_HOUR_NEW").is_between(6, 9)).then(pl.lit("Morning Rush"))
         .when(pl.col("CRASH_HOUR_NEW").is_between(15, 19)).then(pl.lit("Evening Rush"))
         .otherwise(pl.lit("Off-Peak")).alias("TIME_OF_DAY"),
        (pl.col("POSTED_SPEED_LIMIT") >= 40).alias("HIGH_SPEED_ZONE")
    ])
)

print("Data types converted and features engineered")

=== Data Type Handling + Feature Engineering ===
Data types converted and features engineered


# **Filter & Clean Data + Handle Outliers**

In [16]:
print("=== Filtering + Outlier Treatment ===")

clean_lf = clean_lf.filter(
    (pl.col("CRASH_YEAR") >= 2015) &
    (pl.col("POSTED_SPEED_LIMIT") <= 70) &
    (pl.col("LATITUDE").is_between(41.6, 42.05)) &
    (pl.col("LONGITUDE").is_between(-87.95, -87.5))
)

# Outlier handling (clipping)
clean_lf = clean_lf.with_columns([
    pl.col("INJURIES_TOTAL").clip(upper_bound=10).alias("INJURIES_TOTAL"),
    pl.col("NUM_UNITS").clip(upper_bound=15).alias("NUM_UNITS")
])

print("Rows after filtering and outlier handling:",
      clean_lf.select(pl.len()).collect().item())

=== Filtering + Outlier Treatment ===
Rows after filtering and outlier handling: 1059066


# **Aggregate Data**

In [17]:
print("=== Aggregation Summary ===")

agg_df = (
    clean_lf
    .group_by(["CRASH_YEAR", "CRASH_MONTH_NEW", "TIME_OF_DAY"])
    .agg([
        pl.len().alias("CRASH_COUNT"),
        pl.col("INJURIES_TOTAL").sum().alias("TOTAL_INJURIES"),
        pl.col("HAD_FATALITY").sum().alias("FATAL_CRASHES"),
        pl.col("HAD_INJURIES").mean().alias("INJURY_RATE")
    ])
    .sort(["CRASH_YEAR", "CRASH_MONTH_NEW"])
)

display(agg_df.collect().head(15))

=== Aggregation Summary ===


CRASH_YEAR,CRASH_MONTH_NEW,TIME_OF_DAY,CRASH_COUNT,TOTAL_INJURIES,FATAL_CRASHES,INJURY_RATE
i32,i8,str,u32,i64,u32,f64
2015,1,"""Off-Peak""",1,0,0,0.0
2015,1,"""Morning Rush""",1,0,0,0.0
2015,1,"""Evening Rush""",3,0,0,0.0
2015,2,"""Morning Rush""",3,0,0,0.0
2015,2,"""Off-Peak""",1,0,0,0.0
…,…,…,…,…,…,…
2015,7,"""Evening Rush""",4,0,0,0.0
2015,7,"""Morning Rush""",1,0,0,0.0
2015,7,"""Off-Peak""",10,0,0,0.0


# **Validate Data**

In [18]:
final_df = clean_lf.collect()

print("=== Validation Checks ===")
print("No duplicate IDs:", final_df["CRASH_RECORD_ID"].is_unique().all())
print("Date range:", final_df["CRASH_DATE_ONLY"].min(), "to", final_df["CRASH_DATE_ONLY"].max())
print("Lat range:", final_df["LATITUDE"].min(), "—", final_df["LATITUDE"].max())
print("Lon range:", final_df["LONGITUDE"].min(), "—", final_df["LONGITUDE"].max())
print("Any negative injuries?", (final_df["INJURIES_TOTAL"] < 0).any())
print("Memory usage:", round(final_df.estimated_size() / (1024**2), 1), "MB")

=== Validation Checks ===
No duplicate IDs: True
Date range: 2015-01-03 to 2026-06-22
Lat range: 41.64461135864258 — 42.02278137207031
Lon range: -87.9396743774414 — -87.52458953857422
Any negative injuries? False
Memory usage: 572.6 MB


# **Export Clean Parquet**

In [23]:
import shutil

# === FINAL EXPORT - Fixed ===
final_df = clean_lf.collect()

print("=== Final Export ===")
print(f"Final row count: {len(final_df):,}")
print(f"Memory: {round(final_df.estimated_size() / (1024**2), 1)} MB")

data_dir = "/content/data"
os.makedirs(data_dir, exist_ok=True)
parquet_path = f"{data_dir}/chicago_traffic_crashes_cleaned.parquet"

# Remove existing directory or file if it conflicts
if os.path.exists(parquet_path):
    if os.path.isdir(parquet_path):
        shutil.rmtree(parquet_path)      # Delete folder from previous partitioned write
        print(" Removed old partitioned directory")
    else:
        os.remove(parquet_path)
        print(" Removed old file")

# Write as single file (safer for now)
final_df.write_parquet(
    parquet_path,
    compression="zstd",
    row_group_size=50_000,
    statistics=True
)

print(f"Parquet successfully saved!")
print(f"Path: {parquet_path}")
print(f"Size: {os.path.getsize(parquet_path) / (1024**2):.2f} MB")

=== Final Export ===
Final row count: 1,067,469
Memory: 598.5 MB
 Removed old partitioned directory
Parquet successfully saved!
Path: /content/data/chicago_traffic_crashes_cleaned.parquet
Size: 125.14 MB


In [24]:
print("Final checks:")
print("File exists?", os.path.exists(parquet_path))
print("Is it a file?", os.path.isfile(parquet_path))
print("Size (MB):", round(os.path.getsize(parquet_path) / (1024**2), 2))

Final checks:
File exists? True
Is it a file? True
Size (MB): 125.14


In [26]:
from google.colab import files
files.download(parquet_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📋 Cleaning Process Documentation

**Completed Steps:**
- ✅ Loaded raw CSV with Polars
- ✅ Initial inspection (schema, stats, missing values)
- ✅ Removed duplicates using `CRASH_RECORD_ID`
- ✅ Handled missing values (critical drops + imputation)
- ✅ Converted data types (datetime, Int16, Float32)
- ✅ Feature Engineering (time categories, flags, etc.)
- ✅ Filtered invalid/out-of-range data
- ✅ Handled outliers via clipping
- ✅ Created aggregations
- ✅ Validated data quality
- ✅ Exported optimized partitioned Parquet

**Milestone 1 Complete!** Ready for BigQuery loading.